In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

tickers = ['AAPL','AMD','CSCO','GOOG','INTC','JNPR','META','MSFT','NFLX','NVDA','TSLA']
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test:{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": stream_index,
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(tickers):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in tickers:
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(tickers)

Pushed to test:CSCO: {'symbol': 'CSCO', 'timestamp': '2025-04-28T10:48:44.909257+00:00', 'open': 98.79, 'high': 101.57, 'low': 97.76, 'close': 100.53, 'volume': 125, 'trade_count': 44, 'vwap': 99.57}
Pushed to test:AMD: {'symbol': 'AMD', 'timestamp': '2025-04-28T10:48:44.909257+00:00', 'open': 101.68, 'high': 103.4, 'low': 101.19, 'close': 102.51, 'volume': 958, 'trade_count': 27, 'vwap': 102.61}
Pushed to test:INTC: {'symbol': 'INTC', 'timestamp': '2025-04-28T10:48:44.909257+00:00', 'open': 97.71, 'high': 99.81, 'low': 97.32, 'close': 98.84, 'volume': 205, 'trade_count': 26, 'vwap': 98.73}
Pushed to test:JNPR: {'symbol': 'JNPR', 'timestamp': '2025-04-28T10:48:44.909257+00:00', 'open': 100.44, 'high': 103.0, 'low': 99.91, 'close': 102.02, 'volume': 162, 'trade_count': 18, 'vwap': 102.28}
Pushed to test:MSFT: {'symbol': 'MSFT', 'timestamp': '2025-04-28T10:48:44.914425+00:00', 'open': 81.83, 'high': 84.28, 'low': 81.95, 'close': 83.29, 'volume': 895, 'trade_count': 39, 'vwap': 83.65}
Pus